# SmolVLA on the Panda chess env, trained from Quest 3 teleop demos

This notebook does the GPU half of the project:

1. installs the repo and LeRobot,
2. downloads the Quest 3 teleop datasets (only the parquet/metadata, not the videos),
3. renders the sim-verified Panda rollouts (retargeted from the teleop demos, plus board-symmetry augmentations) into a LeRobot dataset,
4. fine-tunes `lerobot/smolvla_base`,
5. evaluates the policy closed-loop in MuJoCo on seen tasks, unseen square pairs and unseen squares.

**Runtime:** GPU. An L4 or A100 (Colab Pro) is recommended; a free T4 works with `BATCH=8` but is slow.
Checkpoints go to Google Drive so a disconnected session can resume.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- settings -------------------------------------------------------------
REPO_URL = "https://github.com/rayrsys/Humanoid-Challenge-.git"   # use https://<token>@github.com/... if the repo is private
BRANCH   = "claude/lucid-lamport-0dndx4"
STEPS    = 10000      # 10k is a good first run; 20k for the final one
BATCH    = 32         # 16 on an L4 if you run out of memory, 8 on a T4
MAX_EPISODES = 0      # 0 = export every sim-verified training rollout
DRIVE_DIR = "/content/drive/MyDrive/humanoid_challenge"

from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

In [ ]:
%cd /content
!rm -rf Humanoid-Challenge-
!git clone -q -b {BRANCH} {REPO_URL}
%cd /content/Humanoid-Challenge-
!pip -q install -e . "lerobot[smolvla]"

In [ ]:
# Teleop data (G1 + Dex3, collected with a Meta Quest 3): parquet + metadata only.
from huggingface_hub import snapshot_download
for side in ("right", "left"):
    snapshot_download(f"Raysolo/fb32-v03-all5-{side}", repo_type="dataset",
                      local_dir=f"data/fb32-v03-all5-{side}",
                      allow_patterns=["data/*", "meta/*", "provenance.json"])
!ls data/*

In [ ]:
# Sanity check: recover the board from the demos and replay a few on the Panda.
!python scripts/calibrate_board.py
!python scripts/replay.py --mode object_centric --limit 10

In [ ]:
# Render the sim-verified rollouts into a LeRobot dataset (EGL on the GPU).
DATASET_ROOT = "/content/lerobot_panda_chess"
limit = f"--limit {MAX_EPISODES}" if MAX_EPISODES else ""
!python scripts/export_lerobot.py --demos artifacts/train_rollouts.jsonl.gz \
    --root {DATASET_ROOT} --repo-id local/panda_chess {limit}

In [ ]:
# Fine-tune SmolVLA. The front/wrist cameras map onto SmolVLA's pretrained camera slots.
OUT = f"{DRIVE_DIR}/smolvla_panda_chess"
!lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id=local/panda_chess --dataset.root={DATASET_ROOT} \
  --rename_map='{{"observation.images.front": "observation.images.camera1", "observation.images.wrist": "observation.images.camera2"}}' \
  --policy.empty_cameras=1 \
  --batch_size={BATCH} --steps={STEPS} --save_freq=2500 \
  --output_dir={OUT} --job_name=smolvla_panda_chess \
  --policy.device=cuda --policy.push_to_hub=false --wandb.enable=false

In [ ]:
# Resume after a disconnect (re-run the settings, clone/install and export cells first).
!lerobot-train --config_path={OUT}/checkpoints/last/pretrained_model/train_config.json --resume=true

In [ ]:
# Closed-loop evaluation in MuJoCo: seen tasks, unseen square pairs, unseen squares.
!python scripts/eval_smolvla.py --ckpt {OUT}/checkpoints/last/pretrained_model --n 30 --video 2 \
    --out {DRIVE_DIR}/eval_smolvla.json
!cp outputs/smolvla_*.mp4 {DRIVE_DIR}/ 2>/dev/null; ls {DRIVE_DIR}

### Notes
* `artifacts/train_rollouts.jsonl.gz` holds the sim-verified action sequences generated by `scripts/generate_demos.py`
  (human demos + board-symmetry augmentations, train split only). Regenerate with
  `python scripts/generate_demos.py --aug-per-demo 4` if you change the retargeting.
* Held-out squares (`b6 c3 d7 e2 f5 g4`) and ~10% of (piece, src, dst) pairs never appear in training; see `chessbot/splits.py`.